## Install Dependencies

Run this cell to install required packages (only needed on first run)


In [1]:
# Install required packages (run this cell on first use)
# This will install all dependencies from requirements.txt
import sys
from pathlib import Path

requirements_file = Path.cwd().parent / "requirements.txt"
if requirements_file.exists():
    print(f"Installing from {requirements_file}...")
    %pip install -q -r {requirements_file}
    print("✓ Dependencies installed!")
else:
    print("⚠️  requirements.txt not found. Installing core packages...")
    %pip install -q telethon networkx pandas pyarrow matplotlib seaborn python-dotenv
    print("✓ Core packages installed!")


Installing from /Users/yuliavistak/Desktop/UCU/Навчання/4 курс/diploma/disinfo_graph/requirements.txt...
ERROR: Invalid requirement: 'курс/diploma/disinfo_graph/requirements.txt'
Hint: It looks like a path. File 'курс/diploma/disinfo_graph/requirements.txt' does not exist.

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✓ Dependencies installed!


## Load Environment Variables

**Important:** Make sure you have created a `.env` file in the project root with your Telegram API credentials before running this cell.


In [2]:
# Load .env file from project root
import sys
from pathlib import Path
from dotenv import load_dotenv

# Get project root (parent of notebooks directory)
project_root = Path.cwd().parent
env_file = project_root / ".env"

# Explicitly load .env file from project root
if env_file.exists():
    load_dotenv(dotenv_path=env_file, override=True)
    print(f"✓ Loaded .env file from: {env_file}")
else:
    print(f"⚠️  .env file not found at: {env_file}")
    print("   Please create .env file in project root with your Telegram API credentials")
    print("   See .env.example for template")

# Add parent directory to path to import disinfograph
sys.path.insert(0, str(project_root))

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Import disinfograph modules
from src.disinfograph.config import (
    get_channel_list,
    get_channel_labels,
    get_channels_csv_path,
    get_telegram_config,
    get_data_paths
)
from src.disinfograph.parser import TelegramParser
from src.disinfograph.graph_builder import build_graph_for_neo4j
from src.disinfograph.similarity_calculator import calculate_and_save_similarity

# Set up plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports successful")


⚠️  .env file not found at: /Users/yuliavistak/Desktop/UCU/Навчання/4 курс/diploma/disinfo_graph/.env
   Please create .env file in project root with your Telegram API credentials
   See .env.example for template


ModuleNotFoundError: No module named 'telethon'

## Step 1: Parse Telegram Channels

Parse a few channels for testing (we'll use the first 2 channels from the CSV)


In [3]:
# Configuration: Set how many channels to test
NUM_TEST_CHANNELS = 2  # Change this number to test with more/fewer channels

# Get configuration
# Note: project_root is already set from the previous cell
paths = get_data_paths(project_root)
telegram_config = get_telegram_config()  # This will now work because .env was loaded
csv_path = get_channels_csv_path(project_root)

# Get all channels from CSV
all_channels = get_channel_list(csv_path)
all_labels = get_channel_labels(csv_path)

# Select channels for testing (first N channels)
test_channels = all_channels[:NUM_TEST_CHANNELS]
test_labels = {ch: all_labels.get(ch.lower(), '') for ch in test_channels}

print(f"📋 Configuration:")
print(f"   Testing with {len(test_channels)} channels (out of {len(all_channels)} total)")
print(f"\n📺 Selected channels:")
for ch in test_channels:
    print(f"   - @{ch} (label: {test_labels.get(ch.lower(), 'N/A')})")


ValueError: TELEGRAM_API_ID and TELEGRAM_API_HASH environment variables must be set.
Create a .env file in the project root with:
  TELEGRAM_API_ID=your_api_id
  TELEGRAM_API_HASH=your_api_hash
Get your credentials from: https://my.telegram.org

## Step 3: Load Data into Pandas

Load the parsed data from Parquet files


In [4]:
# Reload parser module and related modules to ensure we have the latest version
# After refactoring, parser imports from utils, message_utils, and parquet_utils
# So we need to reload all related modules
import importlib
from src.disinfograph import parser as parser_module
from src.disinfograph import utils, message_utils, parquet_utils

# Reload all related modules to ensure latest code
importlib.reload(utils)
importlib.reload(message_utils)
importlib.reload(parquet_utils)
importlib.reload(parser_module)

from src.disinfograph.parser import TelegramParser
from src.disinfograph.date_utils import get_last_n_months_range

# Configuration: Choose parsing mode
# Option 1: Date-based parsing (fetch messages from last N months)
USE_DATE_BASED_PARSING = True  # Set to True to use date-based parsing
MONTHS_BACK = 12  # Number of months to go back (default: 12 months)

# Option 2: Limit-based parsing (fetch N most recent messages)
MESSAGES_PER_CHANNEL = 5  # Number of messages to fetch per channel (only used if USE_DATE_BASED_PARSING = False)

# Other configuration
INCLUDE_RAW_MESSAGE = False  # Set to True to include full raw message data
FETCH_REPLY_PARENT = False  # Set to True to fetch reply parent metadata
ENABLE_JSONL_BACKUP = False  # Set to True to also write JSONL backup files

# Create parser instance
if USE_DATE_BASED_PARSING:
    # Date-based parsing: fetch messages from last N months
    start_date, end_date = get_last_n_months_range(MONTHS_BACK)
    print(f"📅 Date-based parsing: Last {MONTHS_BACK} months")
    print(f"   Date range: {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
    
    parser = TelegramParser(
        api_id=telegram_config["api_id"],
        api_hash=telegram_config["api_hash"],
        session_name=str(paths["session_file"]),
        messages_parquet=paths["messages_parquet"],
        channels_parquet=paths["channels_parquet"],
        channels=test_channels,
        channel_labels=test_labels,
        messages_per_channel=None,  # None means fetch all in date range
        start_date=start_date,
        end_date=end_date,
        include_raw_message=INCLUDE_RAW_MESSAGE,
        fetch_reply_parent=FETCH_REPLY_PARENT,
        output_messages_file=paths["messages_file"] if ENABLE_JSONL_BACKUP else None,
        output_channels_file=paths["channels_file"] if ENABLE_JSONL_BACKUP else None,
    )
else:
    # Limit-based parsing: fetch N most recent messages
    print(f"📊 Limit-based parsing: {MESSAGES_PER_CHANNEL} messages per channel")
    
    parser = TelegramParser(
        api_id=telegram_config["api_id"],
        api_hash=telegram_config["api_hash"],
        session_name=str(paths["session_file"]),
        messages_parquet=paths["messages_parquet"],
        channels_parquet=paths["channels_parquet"],
        channels=test_channels,
        channel_labels=test_labels,
        messages_per_channel=MESSAGES_PER_CHANNEL,
        start_date=None,  # No date filtering
        end_date=None,
        include_raw_message=INCLUDE_RAW_MESSAGE,
        fetch_reply_parent=FETCH_REPLY_PARENT,
        output_messages_file=paths["messages_file"] if ENABLE_JSONL_BACKUP else None,
        output_channels_file=paths["channels_file"] if ENABLE_JSONL_BACKUP else None,
    )

print("✓ Parser configured")
print(f"   Channels: {len(test_channels)}")
if USE_DATE_BASED_PARSING:
    print(f"   Date range: Last {MONTHS_BACK} months")
else:
    print(f"   Messages per channel: {MESSAGES_PER_CHANNEL}")
print(f"   Include raw message: {INCLUDE_RAW_MESSAGE}")
print(f"   Fetch reply parent: {FETCH_REPLY_PARENT}")
print(f"   JSONL backup: {ENABLE_JSONL_BACKUP}")
print("\n💡 To run parsing, uncomment 'parser.run()' in the next cell")

📅 Date-based parsing: Last 12 months
   Date range: 2024-12-16 to 2025-12-11
✓ Parser configured
   Channels: 2
   Date range: Last 12 months
   Include raw message: False
   Fetch reply parent: False
   JSONL backup: False

💡 To run parsing, uncomment 'parser.run()' in the next cell


**Option 1: Run parser directly in notebook** (uncomment `parser.run()` below)

**Option 2: Run parser as a script** (recommended - avoids async issues):
```python
!cd /Users/vittoriadiachenko/PycharmProjects/DisinfoGraph && python scripts/parse_telegram.py
```

**Note:** This requires Telegram API credentials in `.env` file.


In [5]:
# Option 1: Run parser directly in notebook (RECOMMENDED)
# This uses the parser instance configured above
# Uncomment the line below:
# parser.run()

# Option 2: Run parser as a script (alternative method)
# This uses the standalone script which works in any environment
# Uncomment the line below:
!cd /Users/vittoriadiachenko/PycharmProjects/DisinfoGraph && python scripts/parse_telegram.py

# Note: The script uses hardcoded settings (5 messages per channel).
# For date-based parsing or custom settings, use Option 1 (parser.run()) above.

# After parsing, data will be saved to:

# - data/telegram_messages.parquet (primary format - always created)
# - data/telegram_channels.parquet (primary format - always created)
# 
# Optional (if ENABLE_JSONL_BACKUP = True in parser configuration):
# - data/telegram_messages_extended.jsonl
# - data/telegram_channels_metadata.jsonl

✓ Connected to Telegram

=== Fetching messages from @donbassr ===
  📅 Date-based: Fetching messages from 2024-12-16 to 2025-12-11
  Collected 50 messages from @donbassr
  ✓ Initialized Parquet writer for messages
  ✓ Written 100 messages to Parquet so far...
  Collected 100 messages from @donbassr
  Collected 150 messages from @donbassr
  ✓ Written 200 messages to Parquet so far...
  Collected 200 messages from @donbassr
  Collected 250 messages from @donbassr
  ✓ Written 300 messages to Parquet so far...
  Collected 300 messages from @donbassr
  Collected 350 messages from @donbassr
  ✓ Written 400 messages to Parquet so far...
  Collected 400 messages from @donbassr
  Collected 450 messages from @donbassr
  ✓ Written 500 messages to Parquet so far...
  Collected 500 messages from @donbassr
  Collected 550 messages from @donbassr
  ✓ Written 600 messages to Parquet so far...
  Collected 600 messages from @donbassr
  Collected 650 messages from @donbassr
  ✓ Written 700 messages to Par